# 第20章 模型评估与诊断

本 notebook 用一个教学版“类星体 vs 非类星体”任务说明模型评估：混淆矩阵、precision、recall、F1、阈值扫描、ROC/PR 点、交叉验证和校准直觉。

## 学习目标

- 理解为什么准确率不足以评价模型。
- 会计算混淆矩阵、precision、recall、F1。
- 观察阈值变化如何改变模型行为。
- 理解交叉验证评估的是稳定性，不是最终测试分数。
- 初步认识“分数好看”不等于“概率校准良好”。

In [ ]:
import csv
import math
from pathlib import Path

data_path = Path("../../data/small/object_classification_demo.csv")
with data_path.open(newline="", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

for row in rows:
    for key in ("u_g", "g_r", "r_i", "i_z"):
        row[key] = float(row[key])
    row["is_quasar"] = 1 if row["object_class"] == "quasar" else 0

print("loaded rows:", len(rows))
print("positive count:", sum(row["is_quasar"] for row in rows))


## 一个固定训练/测试切分

我们仍然使用稳定的教学切分。评估章节的重点不是追求最好成绩，而是理解不同指标到底在看什么。

In [ ]:
test_indices = {1, 4, 5, 8, 11, 14}
train_rows = [row for index, row in enumerate(rows) if index not in test_indices]
test_rows = [row for index, row in enumerate(rows) if index in test_indices]

print("train:", len(train_rows), "test:", len(test_rows))
print("test labels:", [row["object_class"] for row in test_rows])


## 构造一个教学版类星体得分

我们用训练集中的“类星体中心”和“非类星体中心”构造一个 0 到 1 之间的分数。这个分数更适合用来展示阈值扫描，而不宣称它是严格校准后的概率。

In [ ]:
FEATURE_KEYS = ("u_g", "g_r", "r_i")


def feature_vector(row):
    return tuple(row[key] for key in FEATURE_KEYS)


def centroid(rows_subset):
    return tuple(sum(row[key] for row in rows_subset) / len(rows_subset) for key in FEATURE_KEYS)


def euclidean_distance(a_value, b_value):
    return math.sqrt(sum((ax - bx) ** 2 for ax, bx in zip(a_value, b_value)))


quasar_rows = [row for row in train_rows if row["is_quasar"] == 1]
non_quasar_rows = [row for row in train_rows if row["is_quasar"] == 0]
quasar_center = centroid(quasar_rows)
non_quasar_center = centroid(non_quasar_rows)


def quasar_score(row):
    features = feature_vector(row)
    distance_to_quasar = euclidean_distance(features, quasar_center)
    distance_to_non_quasar = euclidean_distance(features, non_quasar_center)
    return distance_to_non_quasar / (distance_to_quasar + distance_to_non_quasar)


test_scores = [quasar_score(row) for row in test_rows]
[(row["object_id"], row["object_class"], round(score, 3)) for row, score in zip(test_rows, test_scores)]


## 混淆矩阵与基础指标

我们先在一个固定阈值上计算二分类指标。

In [ ]:
def predict_from_threshold(scores, threshold):
    return [1 if score >= threshold else 0 for score in scores]


def confusion_counts(y_true, y_pred):
    tp = sum(1 for truth, pred in zip(y_true, y_pred) if truth == 1 and pred == 1)
    fn = sum(1 for truth, pred in zip(y_true, y_pred) if truth == 1 and pred == 0)
    fp = sum(1 for truth, pred in zip(y_true, y_pred) if truth == 0 and pred == 1)
    tn = sum(1 for truth, pred in zip(y_true, y_pred) if truth == 0 and pred == 0)
    return {"positive": {"positive": tp, "negative": fn}, "negative": {"positive": fp, "negative": tn}}


def precision_score(y_true, y_pred):
    tp = sum(1 for truth, pred in zip(y_true, y_pred) if truth == 1 and pred == 1)
    fp = sum(1 for truth, pred in zip(y_true, y_pred) if truth == 0 and pred == 1)
    return tp / (tp + fp) if (tp + fp) else 0.0


def recall_score(y_true, y_pred):
    tp = sum(1 for truth, pred in zip(y_true, y_pred) if truth == 1 and pred == 1)
    fn = sum(1 for truth, pred in zip(y_true, y_pred) if truth == 1 and pred == 0)
    return tp / (tp + fn) if (tp + fn) else 0.0


def f1_score(y_true, y_pred):
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    return 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0


y_test = [row["is_quasar"] for row in test_rows]
threshold = 0.5
predictions = predict_from_threshold(test_scores, threshold)

print("confusion =", confusion_counts(y_test, predictions))
print("precision =", round(precision_score(y_test, predictions), 3))
print("recall =", round(recall_score(y_test, predictions), 3))
print("f1 =", round(f1_score(y_test, predictions), 3))


## 阈值扫描：precision 和 recall 的交换

阈值越高，模型通常越保守；阈值越低，召回通常更高，但误报可能增加。

In [ ]:
thresholds = [0.2, 0.5, 0.85]
threshold_summary = []
for threshold in thresholds:
    threshold_predictions = predict_from_threshold(test_scores, threshold)
    threshold_summary.append({
        "threshold": threshold,
        "precision": round(precision_score(y_test, threshold_predictions), 3),
        "recall": round(recall_score(y_test, threshold_predictions), 3),
        "f1": round(f1_score(y_test, threshold_predictions), 3),
    })

threshold_summary


## ROC 与 PR 点

这里不画图，而是直接计算几个离散阈值下的 ROC / PR 点，观察不同指标如何联动。

In [ ]:
def false_positive_rate(y_true, y_pred):
    fp = sum(1 for truth, pred in zip(y_true, y_pred) if truth == 0 and pred == 1)
    tn = sum(1 for truth, pred in zip(y_true, y_pred) if truth == 0 and pred == 0)
    return fp / (fp + tn) if (fp + tn) else 0.0


scan_thresholds = [0.0] + sorted({round(score, 3) for score in test_scores}, reverse=True)
roc_points = []
pr_points = []
for threshold in scan_thresholds:
    threshold_predictions = predict_from_threshold(test_scores, threshold)
    recall = recall_score(y_test, threshold_predictions)
    precision = precision_score(y_test, threshold_predictions)
    roc_points.append((round(false_positive_rate(y_test, threshold_predictions), 3), round(recall, 3)))
    pr_points.append((round(recall, 3), round(precision, 3)))

print("roc points =", roc_points)
print("pr points =", pr_points)


## 交叉验证：看稳定性，而不是只看一次切分

为了保持完全可运行，我们用一个最小的 3 折交叉验证来测试教学版 kNN 分类器在三分类任务上的稳定性。

In [ ]:
def classify_knn_multiclass(target_row, training_rows, k_neighbors=3):
    target_features = feature_vector(target_row)
    distances = []
    for row in training_rows:
        distances.append((euclidean_distance(target_features, feature_vector(row)), row["object_class"]))
    distances.sort(key=lambda item: item[0])

    votes = {}
    for _, label in distances[:k_neighbors]:
        votes[label] = votes.get(label, 0) + 1
    return max(sorted(votes), key=lambda label: votes[label])


folds = [
    [0, 3, 6, 9, 12],
    [1, 4, 7, 10, 13],
    [2, 5, 8, 11, 14],
]

cv_scores = []
for fold in folds:
    validation_rows = [rows[index] for index in fold]
    training_rows = [row for index, row in enumerate(rows) if index not in fold]
    predictions = [classify_knn_multiclass(row, training_rows, k_neighbors=3) for row in validation_rows]
    truths = [row["object_class"] for row in validation_rows]
    score = sum(1 for truth, pred in zip(truths, predictions) if truth == pred) / len(truths)
    cv_scores.append(round(score, 3))

print("cv scores =", cv_scores)
print("cv mean =", round(sum(cv_scores) / len(cv_scores), 3))


## 校准直觉：Brier score

下面这个分数提醒我们：一个能排序的分数，并不自动是完美校准的概率。

In [ ]:
def brier_score(y_true, y_prob):
    return sum((truth - prob) ** 2 for truth, prob in zip(y_true, y_prob)) / len(y_true)


print("brier score =", round(brier_score(y_test, test_scores), 4))


## 小结

模型评估不是为了给结果配一个漂亮数字，而是为了判断模型在你的科学目标里是否真的值得信任。混淆矩阵、阈值扫描、ROC/PR、交叉验证和校准，分别回答的是不同层面的问题。